In [1]:
from pathlib import Path
import pandas as pd

In [2]:
SAMPLE_DATA_DIR=Path("/Users/sam/Desktop/projects/llm/llms/data/fineweb_1.25B/train/shard_00000.parquet")

In [3]:
row = 0
df = pd.read_parquet(SAMPLE_DATA_DIR)
text = df["text"][row]
text

'Amazon.com (AMZN) will have a difficult time meeting analyst expectations this quarter given its several significant headwinds. The two most prominent headwinds relate to revenue, namely the imposition of state sales taxes on Amazon e-commerce revenue and the large increase in federal taxes impacting consumer demand. There is another additional key headwind that is rarely discussed in relation to Amazon, but has and will continue to have a significant impact on operating income - the price of gasoline. While it should be no surprise that gasoline impacts shipping costs, this article enables the reader to accurately predict the impact of gasoline price changes on gross margin, operating income and earnings per share.\nWhile Amazon is quite stingy in the investment information it releases to investors (missing are tablet units, impact of sales tax, streaming users, 3P total merchandise value, AWS COGS, etc…), they do let us know their gross and net shipping expenses. Figure 1 charts gro

In [4]:
tokens = text.encode("utf-8")
tokens = list(map(int, tokens))

print(f"text length {len(text)}")
print(f"tokens length {len(tokens)}")

text length 7313
tokens length 7317


In [5]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
top_pair = max(stats, key=stats.get)
print(top_pair)
print(stats.get(top_pair, 0))

(101, 32)
207


In [6]:
def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

tokens2 = merge(tokens, top_pair, len(tokens) + 1)
len(tokens2)

7110

In [7]:
total_tokens = len(tokens)
vocab_size = 8_000
num_merges = vocab_size - total_tokens
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = total_tokens + i
    print(f"merging pair {pair} into new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    

merging pair (101, 32) into new token 7317
merging pair (115, 32) into new token 7318
merging pair (105, 110) into new token 7319
merging pair (116, 32) into new token 7320
merging pair (111, 110) into new token 7321
merging pair (101, 114) into new token 7322
merging pair (101, 110) into new token 7323
merging pair (116, 104) into new token 7324
merging pair (100, 32) into new token 7325
merging pair (114, 101) into new token 7326
merging pair (111, 32) into new token 7327
merging pair (121, 32) into new token 7328
merging pair (97, 110) into new token 7329
merging pair (7319, 103) into new token 7330
merging pair (97, 114) into new token 7331
merging pair (7321, 32) into new token 7332
merging pair (116, 7327) into new token 7333
merging pair (97, 108) into new token 7334
merging pair (97, 115) into new token 7335
merging pair (7330, 32) into new token 7336
merging pair (104, 105) into new token 7337
merging pair (116, 105) into new token 7338
merging pair (7324, 7317) into new token

In [8]:
print(f"total tokens {total_tokens}")
print(f"ids length {len(ids)}")
print(f"compression ratio {len(tokens) / len(ids):.2f}X")

total tokens 7317
ids length 1840
compression ratio 3.98X


In [12]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text


print(decode([128]))

�


In [14]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens
        
print(encode("Hello world!"))

[72, 7894, 7327, 119, 7352, 108, 100, 33]


In [16]:
print(decode(encode("Hello world!")))

Hello world!


In [17]:
text2 = decode(encode(text))
print(text2 == text)

True


In [18]:
valtext = df["text"][row + 1]
valtext2 = decode(encode(valtext))
print(valtext == valtext2)

True
